<a href="https://colab.research.google.com/github/seo55555/dkdltmxl/blob/main/code_05_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# 0. 환경 설정과 4주차 함수 재사용

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 170)
np.set_printoptions(suppress=True)

In [4]:
import os
from google.colab import files
files.upload()

DATA, BOOK = 'knhanes_like_week4.csv', 'codebook_week4.csv'
df = pd.read_csv(DATA)
w  = df['wt'].to_numpy()
ST, PS = df['kstrata'].to_numpy(), df['psu'].to_numpy()

print(f'n = {len(df):,}  |  조사구 {df.psu.nunique()}개  |  층 {df.kstrata.nunique()}개')
print(f'가중치 합 = {w.sum():,.0f}')

Saving codebook_week4.csv to codebook_week4 (1).csv
Saving knhanes_like_week4.csv to knhanes_like_week4 (1).csv
n = 5,367  |  조사구 263개  |  층 18개
가중치 합 = 450,899


---
# 실습 — Table 1 완성


**규칙** — n은 관측 수, %와 평균은 가중치 적용. 층화 열은 **도메인 분석**으로 만듭니다.

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# -------------------------------------------------------------------------
# 1. svymean_domain 함수 정의 (도메인 가중평균 및 표준오차 계산)
# -------------------------------------------------------------------------
def svymean_domain(y, w, ST, PS, dom):
    """
    도메인(dom) 내에서 복합표본 가중평균과 표준오차(SE)를 계산하는 함수
    """
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    ST = np.asarray(ST)
    PS = np.asarray(PS)
    dom = np.asarray(dom, dtype=bool)

    # 도메인 가중치 적용
    w_dom = w * dom
    sum_w = np.sum(w_dom)

    if sum_w == 0:
        return np.nan, np.nan

    # 가중평균 (Estimate)
    mean_est = np.sum(w_dom * y) / sum_w

    # 층화/집단 표본추출에 따른 분산(SE) 계산 (Taylor Series Linearization)
    residuals = dom * (y - mean_est)

    psu_vars = []
    unique_strata = np.unique(ST)

    for st in unique_strata:
        st_mask = (ST == st)
        unique_psus = np.unique(PS[st_mask])
        n_h = len(unique_psus)

        if n_h <= 1:
            continue

        psu_totals = []
        for psu in unique_psus:
            psu_mask = st_mask & (PS == psu)
            psu_totals.append(np.sum(w_dom[psu_mask] * residuals[psu_mask]))

        psu_totals = np.array(psu_totals)
        mean_psu = np.mean(psu_totals)
        var_h = (n_h / (n_h - 1)) * np.sum((psu_totals - mean_psu) ** 2)
        psu_vars.append(var_h)

    var_total = np.sum(psu_vars) / (sum_w ** 2)
    se_est = np.sqrt(var_total)

    return mean_est, se_est

# -------------------------------------------------------------------------
# 2. 노출군 / 비노출군 정의 (도메인 정의)
# -------------------------------------------------------------------------
# smk: 1=현재흡연, 2=과거흡연, 3=비흡연
smk_exp = (df.smk == 1)                        # 노출군 (현재흡연)
smk_unexp = df.smk.isin([2, 3])                # 비노출군 (비현재흡연: 과거흡연 + 비흡연)

DOMS = {
    '전체': np.ones(len(df), dtype=bool),
    '노출군(현재흡연)': smk_exp,
    '비노출군(비현재흡연)': smk_unexp
}

# -------------------------------------------------------------------------
# 3. Table 1 행 생성 함수 (n 열 포함)
# -------------------------------------------------------------------------
def row_cont(label, var):
    """연속형 변수: 관측수 n, 가중 평균 (SE)"""
    valid_mask = ~df[var].isna()
    n_obs = int(df.loc[valid_mask, var].count())
    r = {'특성': label, 'n': f"{n_obs:,}"}

    for name, dom in DOMS.items():
        combined_dom = dom & valid_mask
        th, se = svymean_domain(df[var], w, ST, PS, combined_dom)
        r[name] = f"{th:.1f} ({se:.2f})"
    return r

def row_cat(label, mask):
    """범주형 변수: 관측수 n, 가중 %"""
    mask_arr = np.asarray(mask, dtype=bool)
    valid_mask = ~df[mask.name if hasattr(mask, 'name') and mask.name in df else 'age'].isna() # 결측치 처리
    n_obs = int(np.sum(mask_arr))
    r = {'특성': label, 'n': f"{n_obs:,}"}

    for name, dom in DOMS.items():
        th, se = svymean_domain(mask_arr.astype(float), w, ST, PS, dom)
        r[name] = f"{th * 100:.1f}%"
    return r

# -------------------------------------------------------------------------
# 4. Table 1 데이터프레임 생성 및 출력
# -------------------------------------------------------------------------
t1_data = [
    row_cont('연령 (세)', 'age'),
    row_cat('  65세 이상', df.age >= 65),
    row_cat('남성', df.sex == 1),
    row_cat('읍면 거주', df.urban == 2),
    row_cont('BMI (kg/m²)', 'bmi'),
    row_cat('  비만 (BMI ≥ 25)', df.bmi >= 25),
    row_cont('수축기혈압 (mmHg)', 'sbp'),
    row_cat('고혈압', df.htn == 1),
    row_cat('현재흡연', df.smk == 1),
    row_cat('규칙적 운동', df.exercise == 2)
]

# 변수 확장 반영 (drink, edu, income, chol, glucose 등 필요 시 아래 추가 가능)
if 'chol' in df.columns:
    t1_data.append(row_cont('총콜레스테롤 (mg/dL)', 'chol'))
if 'glucose' in df.columns:
    t1_data.append(row_cont('공복혈당 (mg/dL)', 'glucose'))

t1 = pd.DataFrame(t1_data)

# 'n' 열을 가장 첫 번째 데이터 열로 배치
cols = ['특성', 'n', '전체', '노출군(현재흡연)', '비노출군(비현재흡연)']
t1 = t1[cols]

# 표 출력 및 각주 표시
print("=== Table 1. 대상자의 일반적 특성 (도메인 분석 적용) ===")
display(t1)
print("\n* 각주: n은 관측 수, %는 가중 추정치, 연속형 변수는 가중평균(표준오차)으로 제시함.")
print("* 표본 설계정보: 가중치(wt), 층(kstrata), 집단(psu) 반영 도메인 분석 실시.")

=== Table 1. 대상자의 일반적 특성 (도메인 분석 적용) ===


,특성,n,전체,노출군(현재흡연),비노출군(비현재흡연)
0,연령 (세),"5,367",49.1 (0.16),48.0 (0.39),49.5 (0.20)
1,65세 이상,"1,607",16.8%,14.4%,17.6%
2,남성,"2,719",49.0%,71.5%,41.9%
3,읍면 거주,"2,156",22.9%,23.6%,22.7%
4,BMI (kg/m²),"5,367",23.8 (0.05),23.3 (0.11),24.0 (0.06)
5,비만 (BMI ≥ 25),"2,035",36.5%,30.7%,38.4%
6,수축기혈압 (mmHg),"5,367",122.8 (0.45),124.0 (0.67),122.4 (0.46)
7,고혈압,"1,652",24.4%,27.2%,23.5%
8,현재흡연,"1,295",24.1%,100.0%,0.0%
9,규칙적 운동,"2,238",41.5%,42.1%,41.3%



* 각주: n은 관측 수, %는 가중 추정치, 연속형 변수는 가중평균(표준오차)으로 제시함.
* 표본 설계정보: 가중치(wt), 층(kstrata), 집단(psu) 반영 도메인 분석 실시.
